In [ ]:
!pip install -q "unsloth>=2024.5.3" "transformers>=4.41.0" "datasets>=2.19.0" "accelerate>=0.30.0" "bitsandbytes>=0.43.0" peft


In [ ]:
import os, random, torch, math
import numpy as np
from dataclasses import dataclass
from typing import Dict, List, Any

from datasets import load_dataset, Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)

from peft import LoraConfig
from unsloth import FastLanguageModel

SEED = 3407
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed()
print("CUDA:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:

model_name = "unsloth/mistral-7b-instruct-v0.2"


MAX_SEQ_LEN = 2048

lora_r = 16
lora_alpha = 32
lora_dropout = 0.1
target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = MAX_SEQ_LEN,
    dtype = None,
    load_in_4bit = True,
)


model = FastLanguageModel.get_peft_model(
    model,
    r = lora_r,
    target_modules = target_modules,
    lora_alpha = lora_alpha,
    lora_dropout = lora_dropout,
    bias = "none",
    use_gradient_checkpointing = True,
    random_state = SEED,
)


tokenizer.padding_side = "right"
tokenizer.truncation_side = "right"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Model & tokenizer ready.")


In [ ]:
import json
from datasets import Dataset

with open("/content/ipc_qa.json") as f:
    data = json.load(f)

raw_ds = Dataset.from_list(data)
print(raw_ds, raw_ds[0])


In [ ]:
ds = DatasetDict({"train": raw_ds})

In [ ]:

def build_prompt(question, context="", answer=None):
    if context and isinstance(context, str) and context.strip():
        prompt = (
            "You are a legal assistant. Provide accurate, concise, and well-cited answers when possible.\n\n"
            f"Question: {question}\n"
            f"Context: {context}\n"
            "Answer:"
        )
    else:
        prompt = (
            "You are a legal assistant. Provide accurate, concise, and jurisdiction-aware answers.\n\n"
            f"Question: {question}\n"
            "Answer:"
        )
    if answer is not None:
        return f"{prompt} {answer}".strip()
    return prompt.strip()

def to_text(example):
    q = example.get("question","").strip()
    a = example.get("answer","").strip()
    c = example.get("context","")
    return {"text": build_prompt(q, c, a)}

formatted = ds["train"].map(to_text, remove_columns=ds["train"].column_names)
formatted = formatted.shuffle(seed=SEED)
print(formatted[0])


In [ ]:
split = formatted.train_test_split(test_size=0.05, seed=SEED)
train_ds, val_ds = split["train"], split["test"]

def tok(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding=False,
        return_attention_mask=True,
    )

train_tok = train_ds.map(tok, batched=True, remove_columns=["text"])
val_tok   = val_ds.map(tok, batched=True, remove_columns=["text"])

print(train_tok[0].keys())
print(f"Train size: {len(train_tok)}, Val size: {len(val_tok)}")


In [ ]:

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


In [ ]:
!pip install -q -U transformers


In [ ]:
EPOCHS = 2
BSZ = 2
GRAD_ACCUM = 8
LR = 2e-4
MAX_STEPS = -1
SAVE_STEPS = 200
LOG_STEPS = 20
WARMUP_STEPS = 50

args = TrainingArguments(
    output_dir = "/content/legal-llm",
    num_train_epochs = EPOCHS,
    per_device_train_batch_size = BSZ,
    per_device_eval_batch_size = BSZ,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate = LR,
    warmup_steps = WARMUP_STEPS,
    lr_scheduler_type = "cosine",
    logging_steps = LOG_STEPS,
    eval_strategy = "steps",
    eval_steps = SAVE_STEPS,
    save_steps = SAVE_STEPS,
    save_total_limit = 2,
    bf16 = torch.cuda.is_bf16_supported(),
    fp16 = not torch.cuda.is_bf16_supported(),
    dataloader_num_workers = 2,
    optim = "adamw_8bit",
    max_steps = MAX_STEPS,
    report_to = "none",
    gradient_checkpointing = True,
    logging_first_step = True,
)
print(args)


In [ ]:
def compute_metrics(eval_pred):

    return {}

trainer = Trainer(
    model = model,
    args = args,
    train_dataset = train_tok,
    eval_dataset = val_tok,
    data_collator = collator,
)

train_result = trainer.train()
trainer.save_state()
trainer.save_model("/content/legal-llm/checkpoint-last")
print("Training finished.")


In [ ]:
from torch.utils.data import DataLoader

val_ds = formatted.select(range(min(20, len(formatted))))

sample_loader = DataLoader(val_ds.select(range(min(3, len(val_ds)))), batch_size=1)

for i, ex in enumerate(sample_loader):
    prompt = ex["text"][0]
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    out = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        top_p=0.95,
        temperature=0.7
    )
    print(f"\n--- Example {i+1} ---")
    print(tokenizer.decode(out[0], skip_special_tokens=True))


In [ ]:
ADAPTER_DIR = "/content/legal-llm/lora-adapter"
MERGED_DIR  = "/content/legal-llm/merged"


trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Saved LoRA adapters to:", ADAPTER_DIR)

try:
    merged = trainer.model.merge_and_unload()
    merged.save_pretrained(MERGED_DIR, safe_serialization=True)
    tokenizer.save_pretrained(MERGED_DIR)
    print("Saved MERGED model to:", MERGED_DIR)
except Exception as e:
    print("Merge failed (likely VRAM or base model restrictions). Use adapters:", e)


In [ ]:

base, tok = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = MAX_SEQ_LEN,
    dtype = None,
    load_in_4bit = True,
)
from peft import PeftModel
inf_model = PeftModel.from_pretrained(base, ADAPTER_DIR)
FastLanguageModel.for_inference(inf_model)

def answer(question, context=""):
    prompt = build_prompt(question, context, answer=None)
    inputs = tok(prompt, return_tensors="pt").to("cuda")
    out = inf_model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=True,
        top_p=0.9,
        temperature=0.7,
        eos_token_id=tok.eos_token_id,
    )
    return tok.decode(out[0], skip_special_tokens=True)

print(answer("What is res judicata under CPC?"))


In [ ]:

print(answer("What is 'Man', 'Woman', 'Person', and 'Public' according to the Indian Penal Code?"))